# Week 05: Model of the solar cycle



## 1) Setup (run this first)

This project is designed to run on Google Colab. Run the setup cell below to install dependencies and configure the environment.

In [ ]:
# Standard ButterflAI Colab setup — always run this first
import os
import subprocess
import sys

# Keep this path if working in Colab
# repo_path = "/content/butterflai"

# Use this path if working locally
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(
        ["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path],
        check=True,
    )
else:
    subprocess.run(["git", "-C", repo_path, "pull"], check=True)

sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()


## 2) Load the Sunspot Group Catalog

We use the provided  file, which contains daily sunspot group measurements from 1825–2023.

In [ ]:
import pandas as pd
from pathlib import Path

data_path = Path(repo_path) / 'data' / 'composite_sunspot_groups_daily_measurements_10_23.csv'
df = pd.read_csv(data_path, parse_dates=[[0, 1, 2]], keep_date_col=False)
df.rename(columns={"year_month_day": "date"}, inplace=True)

# Keep only rows with valid latitude (sunspot groups)
df = df[df["latitude"].notna()].copy()

# Quick sanity check
df[["date", "latitude", "CYCLE"]].head()


## 3) Plot the Butterfly Diagram

We plot emergence latitude vs. date. Points are colored by hemisphere (north vs south).

In [ ]:
import matplotlib.pyplot as plt

df["hemisphere"] = df["latitude"].apply(lambda v: "north" if v >= 0 else "south")
colors = {"north": "tab:red", "south": "tab:blue"}

fig, ax = plt.subplots(figsize=(10, 5))
for hemi, group in df.groupby("hemisphere"):
    ax.scatter(group["date"], group["latitude"], s=2, c=colors[hemi], label=hemi, alpha=0.4)

ax.set_title("Solar Butterfly Diagram (Sunspot Emergence Latitude vs Time)")
ax.set_xlabel("Date")
ax.set_ylabel("Latitude (degrees)")
ax.legend(title="Hemisphere")
ax.set_ylim(-45, 45)
plt.show()


## 4) Exercise: Visualize Individual Solar Cycles

The butterfly diagram shows all cycles overlaid. Create a visualization that focuses on a single solar cycle (e.g., cycle 24, which is recent and well-observed).

**Task:** 
- Filter the data to a specific cycle (use the  column).
- Plot the butterfly diagram for just that cycle.
- Experiment with different cycles and note any differences in shape or amplitude.

**Hint:** You can use the cycle column to filter.


In [ ]:
# Task 4: Visualize a single solar cycle
cycle_number = 24  # Change this to explore other cycles

df_cycle = df[df["CYCLE"] == cycle_number].copy()

fig, ax = plt.subplots(figsize=(10, 5))
for hemi, group in df_cycle.groupby("hemisphere"):
    ax.scatter(group["date"], group["latitude"], s=4, c=colors[hemi], label=hemi, alpha=0.5)

ax.set_title(f"Solar Butterfly Diagram — Cycle {cycle_number}")
ax.set_xlabel("Date")
ax.set_ylabel("Latitude (degrees)")
ax.legend(title="Hemisphere")
ax.set_ylim(-45, 45)
ax.axhline(0, color="k", linewidth=0.5, linestyle="--")
plt.tight_layout()
plt.show()

print(f"Cycle {cycle_number}: {df_cycle['date'].min().date()} → {df_cycle['date'].max().date()}")
print(f"  N groups: {len(df_cycle):,}")
print(f"  Lat range: [{df_cycle['latitude'].min():.1f}°, {df_cycle['latitude'].max():.1f}°]")


## 5) Exercise: Calculate Mean Latitudinal Paths

The "wings" of the butterfly diagram show the average latitude of sunspot emergence over time for each hemisphere.

**Task:**
- Group the data by hemisphere and calculate the mean latitude for each month/year.
- Plot the mean latitudinal path for the northern and southern hemispheres separately.
- Discuss: What does this tell us about the solar cycle's evolution?

**Hint:** Use  with  and date aggregation (e.g., resample by year).


In [ ]:
# Task 5: Calculate and plot mean latitudinal paths by cycle

# Generate a colormap for cycles
import matplotlib.cm as cm
import numpy as np

# Filter out rows with missing CYCLE values
cycles = sorted(df["CYCLE"].dropna().unique())
n_cycles = len(cycles)
cmap = cm.get_cmap("tab20", n_cycles)
cycle_colors = {cyc: cmap(i) for i, cyc in enumerate(cycles)}

fig, ax = plt.subplots(figsize=(14, 3))

# Plot butterfly diagram with each cycle in a different color
for cyc in cycles:
    df_cyc = df[df["CYCLE"] == cyc]
    ax.scatter(df_cyc["date"], df_cyc["latitude"], s=2, 
               c=[cycle_colors[cyc]], label=f"Cycle {int(cyc)}", 
               alpha=0.3, edgecolors="none")

# Calculate and overplot yearly mean latitude for each cycle and hemisphere
df["year"] = df["date"].dt.year

for cyc in cycles:
    df_cyc_all = df[df["CYCLE"] == cyc]
    
    # Northern hemisphere
    df_cyc_north = df_cyc_all[df_cyc_all["hemisphere"] == "north"]
    if len(df_cyc_north) > 0:
        yearly_means_north = df_cyc_north.groupby("year")["latitude"].mean().sort_index()
        year_dates = yearly_means_north.index.astype(str)
        year_dates_dt = pd.to_datetime(year_dates + "-01-01")
        ax.plot(year_dates_dt, yearly_means_north.values, color="black", 
                linewidth=1.5, linestyle="-", alpha=0.8)
    
    # Southern hemisphere
    df_cyc_south = df_cyc_all[df_cyc_all["hemisphere"] == "south"]
    if len(df_cyc_south) > 0:
        yearly_means_south = df_cyc_south.groupby("year")["latitude"].mean().sort_index()
        year_dates = yearly_means_south.index.astype(str)
        year_dates_dt = pd.to_datetime(year_dates + "-01-01")
        ax.plot(year_dates_dt, yearly_means_south.values, color="black", 
                linewidth=1.5, linestyle="--", alpha=0.8)

ax.set_title("Solar Butterfly Diagram with Yearly Mean Latitudes by Hemisphere")
ax.set_xlabel("Date")
ax.set_ylabel("Latitude (degrees)")
ax.set_ylim(-45, 45)
ax.axhline(0, color="k", linewidth=0.5, linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()


The cell below does the same, but for a single cycle

In [ ]:
# Bonus: Visualize a single cycle with its mean latitudinal paths
cycle_number = 24  # Change this to explore other cycles

df_cyc_bonus = df[df["CYCLE"] == cycle_number].copy()

fig, ax = plt.subplots(figsize=(12, 12))

# Plot butterfly diagram for this cycle with different colors per hemisphere
for hemi, color in [("north", "tab:red"), ("south", "tab:blue")]:
    df_hemi = df_cyc_bonus[df_cyc_bonus["hemisphere"] == hemi]
    ax.scatter(df_hemi["date"], df_hemi["latitude"], s=4, c=color, label=hemi, alpha=0.4)

# Overplot yearly mean paths for this cycle
df_cyc_all = df[df["CYCLE"] == cycle_number]

# Northern hemisphere mean
df_cyc_north = df_cyc_all[df_cyc_all["hemisphere"] == "north"]
if len(df_cyc_north) > 0:
    yearly_means_north = df_cyc_north.groupby("year")["latitude"].mean().sort_index()
    year_dates = yearly_means_north.index.astype(str)
    year_dates_dt = pd.to_datetime(year_dates + "-01-01")
    ax.plot(year_dates_dt, yearly_means_north.values, color="black", 
            linewidth=2, linestyle="-", alpha=0.9, label="North mean")

# Southern hemisphere mean
df_cyc_south = df_cyc_all[df_cyc_all["hemisphere"] == "south"]
if len(df_cyc_south) > 0:
    yearly_means_south = df_cyc_south.groupby("year")["latitude"].mean().sort_index()
    year_dates = yearly_means_south.index.astype(str)
    year_dates_dt = pd.to_datetime(year_dates + "-01-01")
    ax.plot(year_dates_dt, yearly_means_south.values, color="black", 
            linewidth=2, linestyle="--", alpha=0.9, label="South mean")

ax.set_title(f"Solar Cycle {cycle_number} with Yearly Mean Latitudinal Paths")
ax.set_xlabel("Date")
ax.set_ylabel("Latitude (degrees)")
ax.legend(title="Hemisphere", loc="upper right")
ax.set_ylim(-45, 45)
ax.axhline(0, color="k", linewidth=0.5, linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

print(f"Cycle {cycle_number}: {df_cyc_bonus['date'].min().date()} → {df_cyc_bonus['date'].max().date()}")
print(f"  N groups: {len(df_cyc_bonus):,}")
print(f"  Lat range: [{df_cyc_bonus['latitude'].min():.1f}°, {df_cyc_bonus['latitude'].max():.1f}°]")


## 6) Empirical Distribution for a Single Year

Before we can model the butterfly diagram statistically, we need to understand what the **distribution of emergence latitudes** looks like at a single moment in time.

**Task:**
- Pick a year (e.g. 1960) and a hemisphere (north or south), and filter the data to that year and hemisphere.
- Plot a **histogram** of emergence latitudes (density-normalized so it compares to a PDF).
- Add vertical lines marking the **median**, **Q1**, and **Q3**, and shade the IQR.
- What shape does the distribution have? Is it symmetric? Try switching to the other hemisphere and try different years — does the shape change?


In [ ]:
# Task 6: Empirical distribution of emergence latitudes for a single year
import numpy as np

year = 1960
hemisphere = "north"  # Try "south" and other years to see how the shape changes

# Filter to the chosen year and hemisphere
mask = (df["year"] == year) & (df["hemisphere"] == hemisphere)
lats = df.loc[mask, "latitude"].values

# Compute quartiles
q1, median, q3 = np.percentile(lats, [25, 50, 75])
iqr = q3 - q1

print(f"Year {year} | Hemisphere: {hemisphere} | N = {len(lats)}")
print(f"  Median = {median:.1f}°   Q1 = {q1:.1f}°   Q3 = {q3:.1f}°   IQR = {iqr:.1f}°")

fig, ax = plt.subplots(figsize=(8, 4))

# Density-normalised histogram so the y-axis is a probability density
ax.hist(lats, bins=20, density=True, color="tab:red", alpha=0.6, edgecolor="white",
        label="Empirical density")

# Shade the IQR
ax.axvspan(q1, q3, color="tab:orange", alpha=0.25, label=f"IQR [{q1:.1f}°, {q3:.1f}°]")

# Vertical lines for the three quartiles
ax.axvline(q1,     color="tab:orange", linestyle="--", linewidth=1.5, label=f"Q1 = {q1:.1f}°")
ax.axvline(median, color="black",      linestyle="-",  linewidth=2,   label=f"Median = {median:.1f}°")
ax.axvline(q3,     color="tab:orange", linestyle=":",  linewidth=1.5, label=f"Q3 = {q3:.1f}°")

ax.set_title(f"Emergence latitude distribution — {year}, {hemisphere} hemisphere")
ax.set_xlabel("Latitude (degrees)")
ax.set_ylabel("Probability density")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 7) Fitting an Analytical Distribution

An empirical histogram is useful but noisy. A fitted **analytical distribution** gives us a smooth, compact description of the data — and forces us to ask whether our chosen family of curves actually matches reality.

**Task:**
- Using the same year and hemisphere as Task 6, fit a **Normal (Gaussian) distribution** to the latitude data and overlay the fitted PDF on the histogram.
- Does it fit well? Where does it agree with the data and where does it fail?
- The Gaussian has two parameters (μ, σ). What do they represent physically for sunspot emergence?

**Hint:** `scipy.stats.norm.fit(data)` returns `(mu, sigma)`. Use `norm.pdf(x, mu, sigma)` to evaluate the PDF on a grid of x values.


In [ ]:
# Task 7: Fit a Normal distribution and overlay on the histogram
from scipy.stats import norm

# Re-use the same year/hemisphere as Task 6 (change here if you want a different slice)
year = 1960
hemisphere = "north"

mask = (df["year"] == year) & (df["hemisphere"] == hemisphere)
lats = df.loc[mask, "latitude"].values

# --- Fit ---
mu, sigma = norm.fit(lats)
print(f"Fitted Normal:  μ = {mu:.2f}°   σ = {sigma:.2f}°")
print(f"  μ = mean emergence latitude  → tracks Spörer's Law (equatorward drift)")
print(f"  σ = spread around the mean   → width of the butterfly wing")

# --- Plot ---
x_grid = np.linspace(lats.min() - 5, lats.max() + 5, 300)

fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(lats, bins=20, density=True, color="tab:red", alpha=0.5, edgecolor="white",
        label="Empirical density")

ax.plot(x_grid, norm.pdf(x_grid, mu, sigma), color="black", linewidth=2,
        label=f"Fitted Normal  μ={mu:.1f}°, σ={sigma:.1f}°")

ax.axvline(mu, color="black", linestyle="--", linewidth=1, alpha=0.7)

ax.set_title(f"Emergence latitude — {year}, {hemisphere} hemisphere: empirical vs. Gaussian fit")
ax.set_xlabel("Latitude (degrees)")
ax.set_ylabel("Probability density")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 8) Distribution Evolution Through a Solar Cycle

A single year's distribution is a snapshot. Over the course of a solar cycle, the distribution **drifts toward the equator** as the cycle ages — this is **Spörer's Law**, one of the most important empirical regularities in solar physics.

**Task:**
- Pick a solar cycle (e.g. cycle 24) and a hemisphere, and filter the data to both.
- For each year in that cycle, compute a **KDE (kernel density estimate)** of emergence latitudes.
- Plot all the KDEs overlaid, colored by year so the time progression is clear.
- Do you see the distributions shifting toward the equator as the cycle progresses? Does the width of the distribution change too?

**Hint:** `scipy.stats.gaussian_kde` fits a smooth density to an array of values. The `bw_method` parameter controls smoothing — try values between 0.2 and 0.5.


In [ ]:
# Task 8: KDE evolution through a solar cycle — butterfly diagram + per-year KDE profiles
import matplotlib.dates as mdates
from scipy.stats import gaussian_kde

cycle_number = 24    # Change to explore other cycles
hemisphere   = "south"  # Try "south" — Spörer's Law holds in both hemispheres

# Filter to cycle + hemisphere
mask = (df["CYCLE"] == cycle_number) & (df["hemisphere"] == hemisphere)
df_cyc_hemi = df[mask].copy()

years_in_cycle = sorted(df_cyc_hemi["year"].unique())
n_years = len(years_in_cycle)

# Colour map: early years → purple, late years → yellow (viridis is perceptually uniform
# and appropriate for a strictly increasing quantity like time)
cmap = plt.get_cmap("viridis", n_years)

# Latitude grid for evaluating KDEs (stay within the Spörer zone)
lat_grid = np.linspace(0, 45, 300) if hemisphere == "north" else np.linspace(-45, 0, 300)

# How wide (in days) should the tallest KDE peak be?
# ~250 days ≈ 2/3 of a year — wide enough to read, narrow enough not to overlap badly
kde_width_days = 250

fig, ax = plt.subplots(figsize=(12, 5))

# --- Background: butterfly diagram scatter ---
scatter_color = "tab:red" if hemisphere == "north" else "tab:blue"
ax.scatter(df_cyc_hemi["date"], df_cyc_hemi["latitude"],
           s=2, color=scatter_color, alpha=0.25, zorder=1, label="Sunspot groups")

# --- Foreground: per-year KDE profiles drawn vertically ---
for i, yr in enumerate(years_in_cycle):
    yr_lats = df_cyc_hemi.loc[df_cyc_hemi["year"] == yr, "latitude"].values
    if len(yr_lats) < 5:          # skip years with too few observations
        continue

    kde = gaussian_kde(yr_lats, bw_method=0.3)
    kde_vals = kde(lat_grid)

    # Normalise so the peak reaches kde_width_days on the x-axis
    kde_scaled = kde_vals / kde_vals.max() * kde_width_days

    # Anchor each profile at July 1 of that year
    center = pd.Timestamp(f"{int(yr)}-07-01")
    x_curve  = [center + pd.Timedelta(days=float(v)) for v in kde_scaled]
    x_anchor = [center] * len(lat_grid)

    color = cmap(i)
    ax.plot(x_curve, lat_grid, color=color, linewidth=1.8, alpha=0.9, zorder=3)
    ax.fill_betweenx(lat_grid, x_anchor, x_curve,
                     color=color, alpha=0.20, zorder=2)
    # Thin vertical baseline at the anchor date
    ax.axvline(center, color=color, linewidth=0.5, alpha=0.4, zorder=1)

    # Horizontal line at the yearly median latitude, extending to the KDE curve edge
    median_lat = np.median(yr_lats)
    kde_at_median = float(kde(np.array([median_lat]))[0])
    kde_at_median_scaled = kde_at_median / kde_vals.max() * kde_width_days
    x_median_end = center + pd.Timedelta(days=kde_at_median_scaled)
    ax.plot([center, x_median_end], [median_lat, median_lat],
            color=color, linewidth=2, linestyle="--", alpha=1.0, zorder=4)

# --- Colour bar (time axis) ---
sm = plt.cm.ScalarMappable(cmap="viridis",
                            norm=plt.Normalize(vmin=years_in_cycle[0],
                                               vmax=years_in_cycle[-1]))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label("Year")

ax.set_title(
    f"Cycle {cycle_number} ({hemisphere}) — butterfly diagram with yearly KDE profiles\n"
    f"KDEs drift equatorward over time → Spörer's Law  |  dashed line = yearly median"
)
ax.set_xlabel("Date")
ax.set_ylabel("Latitude (degrees)")
ax.set_ylim((0, 45) if hemisphere == "north" else (-45, 0))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

# --- Numeric summary: KDE peak latitude per year (tracks Spörer drift) ---
print(f"Cycle {cycle_number} ({hemisphere}) — yearly median and KDE peak latitudes:")
for i, yr in enumerate(years_in_cycle):
    yr_lats = df_cyc_hemi.loc[df_cyc_hemi["year"] == yr, "latitude"].values
    if len(yr_lats) < 5:
        continue
    kde = gaussian_kde(yr_lats, bw_method=0.3)
    peak_lat   = lat_grid[np.argmax(kde(lat_grid))]
    median_lat = np.median(yr_lats)
    print(f"  {int(yr)}: median = {median_lat:.1f}°   KDE peak = {peak_lat:.1f}°")

---
## Week 03 Exercises: From Snapshots to a Parametric Wing Model

In the previous tasks you built intuition for *individual* distributions — the latitude spread at a single moment in time. Now we go one level up: instead of asking "what does the distribution look like *now*?", we ask **"how does the distribution evolve over an entire cycle?"**

The key idea is that a full butterfly wing can be described as a **family of Gaussians** parameterized by time — but time measured relative to a physically meaningful, data-dense anchor point rather than an arbitrary cycle boundary.

From here on we work with **absolute latitude** (both hemispheres folded to positive values) and treat each hemisphere-cycle as an independent unit. This doubles our sample size and makes the analysis symmetric.

The three exercises below build a parametric model step by step.

## 9) Exercise: Standardizing Time to the 15° latitude Crossing

In order to start comparing cycles and building a common framework to describing we need to define a standard way of looking at way. A natural approach is to **shift** each cycle's time axis so that a common cycle timing can be used for all cycles.

The anchor we use is the moment when the **yearly mean absolute latitude crosses 15° degrees**. 

The resulting time coordinate τ = date − t₀ is in physical units (years). 

**Task:**
- Add an `abs_latitude` column to `df` (absolute value of latitude). All subsequent analysis uses this column.
- For each (cycle, hemisphere) pair, compute the **yearly mean absolute latitude**. Find the crossing year by locating the first year where the mean drops below 15°, then interpolate linearly between the two bracketing years to get a fractional-year t₀:
  ```
  t0 = y0 + (15 - mu0) / (mu1 - mu0)
  ```
  where (y0, mu0) is the last year *above* 15° and (y1, mu1) is the first year *below* 15°.
- Add a `tau` column to `df`: for each row, `tau = (date.dt.year + date.dt.dayofyear / 365.25) − t0`, using the t₀ for that row's (cycle, hemisphere).
  - *Hint:* Build a lookup dict `{(cycle, hemi): t0}` first, then map it onto `df`.
- Plot absolute latitude vs. τ for **all** hemisphere-cycles on a single axis, colored by cycle number.
- **Discussion:** Look at the region τ > 0 (post-crossing, decaying phase). Do the wings from different cycles align with each other? Now look at τ < 0 (rising phase). Is the alignment as tight? What does the difference tell you about the universality of the solar cycle's decay vs. its rise?


In [ ]:
# Task 9: Standardizing time to the 15° latitude crossing
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Step 1: Add absolute latitude column
df["abs_latitude"] = df["latitude"].abs()

# Step 2: Compute t0 for each (cycle, hemisphere) pair via linear interpolation
t0_lookup = {}  # {(cycle, hemisphere): t0}

for (cyc, hemi), group in df.groupby(["CYCLE", "hemisphere"]):
    yearly_mean = group.groupby("year")["abs_latitude"].mean().sort_index()
    years = yearly_mean.index.values
    means = yearly_mean.values

    below_mask = means < 15.0
    if not below_mask.any() or below_mask.all():
        continue  # never crosses 15°, or always below — skip

    idx1 = np.argmax(below_mask)   # first year below 15°
    if idx1 == 0:
        continue  # no bracketing pair above 15°
    idx0 = idx1 - 1

    y0, mu0 = years[idx0], means[idx0]
    y1, mu1 = years[idx1], means[idx1]

    # Linear interpolation to fractional year of the 15° crossing
    t0 = y0 + (15.0 - mu0) / (mu1 - mu0)
    t0_lookup[(cyc, hemi)] = t0

# Step 3: Add decimal_year and tau columns
df["decimal_year"] = df["date"].dt.year + df["date"].dt.dayofyear / 365.25

def get_t0(row):
    return t0_lookup.get((row["CYCLE"], row["hemisphere"]), np.nan)

df["t0"]  = df.apply(get_t0, axis=1)
df["tau"] = df["decimal_year"] - df["t0"]

# Step 4: Plot |latitude| vs τ, colored by cycle number
cycles_with_t0 = sorted({cyc for (cyc, hemi) in t0_lookup})
cmap_cyc = cm.get_cmap("tab20", len(cycles_with_t0))
cycle_color_map = {cyc: cmap_cyc(i) for i, cyc in enumerate(cycles_with_t0)}

fig, ax = plt.subplots(figsize=(12, 5))

for (cyc, hemi), group in df.groupby(["CYCLE", "hemisphere"]):
    if (cyc, hemi) not in t0_lookup:
        continue
    ax.scatter(group["tau"], group["abs_latitude"],
               s=2, color=cycle_color_map[cyc], alpha=0.2, edgecolors="none")

ax.axvline(0, color="black", linewidth=1.5, linestyle="--", label="τ = 0  (15° crossing)")
ax.axhline(15, color="gray", linewidth=1, linestyle=":", alpha=0.6)
ax.set_xlabel("τ (years relative to 15° crossing)")
ax.set_ylabel("|Latitude| (degrees)")
ax.set_title("All hemisphere-cycles aligned at the 15° mean latitude crossing")
ax.set_ylim(0, 45)
ax.legend()

sm = plt.cm.ScalarMappable(cmap="tab20",
                            norm=plt.Normalize(vmin=min(cycles_with_t0),
                                               vmax=max(cycles_with_t0)))
sm.set_array([])
fig.colorbar(sm, ax=ax, pad=0.02, label="Cycle number")
plt.tight_layout()
plt.show()

print(f"t₀ computed for {len(t0_lookup)} hemisphere-cycle pairs.")
print(f"τ range in dataset: [{df['tau'].min():.1f}, {df['tau'].max():.1f}] years")


## 10) Exercise: Fitting the mean path μ(τ) of a Single Hemisphere-Cycle

In Task 7 you fitted one Gaussian per *year*. Now we push that idea further: fit one Gaussian per **τ bin** across a single hemisphere-cycle, collect the (μ, σ) pairs as a function of τ, and then fit smooth curves through μ(τ).

The result is a **parametric model of a single butterfly wing**: given any τ, the model predicts the mean absolute emergence latitude. Later we will look at the spread.

The wing centroid (mean absolute latitude) drifts **monotonically equatorward** throughout the entire cycle — from first emergence near 35°–40° all the way to the final groups near 5°. This means a single smooth function can describe the full trajectory.

The appropriate model is an **exponential decay**:

$$\mu(\tau) = a \cdot \exp\!\left(-\frac{\tau}{b}\right)$$

```python
def exp_decay(tau, a, b):
    return a * np.exp(-tau / b)
```

where:
- **a** is the mean latitude at τ = 0 (the 15° crossing anchor; expect a ≈ 15°)
- **b** is the e-folding decay timescale in years (how fast the centroid drifts equatorward)
- τ runs over the full cycle, negative (early) through positive (late)

**Task:**
- Pick a single hemisphere-cycle (e.g. cycle 23, northern hemisphere). Filter `df` to that pair.
- Divide the τ axis into `N_bins = 20` equal-width bins spanning the full τ range. For each bin, fit a Gaussian to `abs_latitude` using `scipy.stats.norm.fit`. Extract μ and σ. Skip bins with fewer than 10 observations.
- Use `scipy.optimize.curve_fit` with `exp_decay` to fit μ(τ). Overlay the fitted curve on a scatter plot of (τ bin center, μ).
- **Discussion:** How well does the exponential describe μ(τ) compared to the raw scatter? 

In [ ]:
# Task 10: Fitting the mean path μ(τ) for a single hemisphere-cycle
# Requires Task 9 to have been run (uses df["tau"], df["abs_latitude"])
from scipy.stats import norm as sp_norm
from scipy.optimize import curve_fit

cycle_sel = 24
hemi_sel  = "south"

mask   = (df["CYCLE"] == cycle_sel) & (df["hemisphere"] == hemi_sel) & df["tau"].notna()
df_sel = df[mask].copy()

# --- Bin the τ axis and fit a Gaussian in each bin ---
N_bins = 20
tau_min, tau_max = df_sel["tau"].min(), df_sel["tau"].max()
bins        = np.linspace(tau_min, tau_max, N_bins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

bin_tau, bin_mu, bin_sigma = [], [], []

for i in range(N_bins):
    in_bin   = (df_sel["tau"] >= bins[i]) & (df_sel["tau"] < bins[i + 1])
    lats_bin = df_sel.loc[in_bin, "abs_latitude"].values
    if len(lats_bin) < 10:
        continue
    mu_f, sigma_f = sp_norm.fit(lats_bin)
    bin_tau.append(bin_centers[i])
    bin_mu.append(mu_f)
    bin_sigma.append(sigma_f)

bin_tau   = np.array(bin_tau)
bin_mu    = np.array(bin_mu)
bin_sigma = np.array(bin_sigma)

# --- Exponential decay model and fit ---
def exp_decay(tau, a, b):
    return a * np.exp(-tau / b)

popt_mu, _ = curve_fit(exp_decay, bin_tau, bin_mu, p0=[15.0, 5.0])
a_mu, b_mu = popt_mu

print(f"μ(τ) fit — Cycle {cycle_sel} ({hemi_sel})")
print(f"  a = {a_mu:.2f}°   (mean latitude at τ = 0, the 15° crossing anchor)")
print(f"  b = {b_mu:.2f} yr (e-folding decay timescale; larger → slower equatorward drift)")

# --- Plot ---
tau_grid = np.linspace(tau_min, tau_max, 300)

fig, ax = plt.subplots(figsize=(10, 5))

ax.scatter(df_sel["tau"], df_sel["abs_latitude"],
           s=2, color="tab:blue", alpha=0.2, label="Sunspot groups")
ax.scatter(bin_tau, bin_mu, color="black", s=50, zorder=5, label="Bin mean μ")
ax.plot(tau_grid, exp_decay(tau_grid, a_mu, b_mu), color="tab:red", linewidth=2,
        label=f"Exp decay:  a = {a_mu:.1f}°,  b = {b_mu:.1f} yr")

ax.axvline(0, color="gray", linewidth=1, linestyle="--", alpha=0.7, label="τ = 0")
ax.set_xlabel("τ (years relative to 15° crossing)")
ax.set_ylabel("|Latitude| (degrees)")
ax.set_title(f"Cycle {cycle_sel} ({hemi_sel}) — mean path μ(τ) with exponential fit")
ax.set_ylim(0, 45)
ax.legend()
plt.tight_layout()
plt.show()


## 11) Exercise: Fitting the spread σ(τ) a Single Hemisphere-Cycle

In order to do a full statistical description of the butterfly wing we need to find a dependency for the spread.   This is harder than the mean path.   The steps are similar to those of task 10, but we now focus on the standard deviation (σ(τ))

**Task:**
- Pick a single hemisphere-cycle (e.g. cycle 23, northern hemisphere). Filter `df` to that pair.
- Divide the τ axis into `N_bins = 20` equal-width bins spanning the full τ range. For each bin, fit a Gaussian to `abs_latitude` using `scipy.stats.norm.fit`. Extract μ and σ. Skip bins with fewer than 10 observations.
- Plot the standard deviation (σ) as a function of tau.
- Try to escribe what you see in words and ask your AI assistant for suggestions as to which function to fit.
- Fit it.
- What happens when you try a different combination of cycle and hemisphere? does it still work?

In [ ]:

# Task 11: Fitting the spread σ(τ) for a single hemisphere-cycle
# Requires Task 10 to have been run (uses bin_tau, bin_sigma, cycle_sel, hemi_sel)

# --- Inspect the raw σ(τ) scatter first ---
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(bin_tau, bin_sigma, color="black", s=50, zorder=5, label="Bin σ")
ax.axvline(0, color="gray", linewidth=1, linestyle="--", alpha=0.7, label="τ = 0")
ax.set_xlabel("τ (years relative to 15° crossing)")
ax.set_ylabel("σ (degrees)")
ax.set_title(f"Cycle {cycle_sel} ({hemi_sel}) — spread σ(τ) (inspect before fitting)")
ax.legend()
plt.tight_layout()
plt.show()

# Observation: σ(τ) also decays roughly exponentially with τ, similar to μ(τ).
# Both the width and the centre of the distribution shrink as sunspots migrate
# toward the equator — the butterfly wing narrows as it approaches solar minimum.
# An exponential decay is therefore an appropriate model for σ(τ) as well.

# --- Fit σ(τ) with an exponential decay ---
from scipy.optimize import curve_fit

def exp_decay(tau, a, b):
    return a * np.exp(-tau / b)

popt_sigma, _ = curve_fit(exp_decay, bin_tau, bin_sigma, p0=[8.0, 5.0])
a_sigma, b_sigma = popt_sigma

print(f"σ(τ) fit — Cycle {cycle_sel} ({hemi_sel})")
print(f"  a = {a_sigma:.2f}°   (spread at τ = 0)")
print(f"  b = {b_sigma:.2f} yr (e-folding timescale for the width)")
print(f"\n  σ/μ ratio at τ=0: {a_sigma/a_mu:.2f}  "
      f"— the wing is ~{a_sigma/a_mu*100:.0f}% as wide as its mean latitude")

tau_grid_11 = np.linspace(bin_tau.min(), bin_tau.max(), 300)

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(bin_tau, bin_sigma, color="black", s=50, zorder=5, label="Bin σ")
ax.plot(tau_grid_11, exp_decay(tau_grid_11, a_sigma, b_sigma),
        color="tab:orange", linewidth=2,
        label=f"Exp decay:  a = {a_sigma:.1f}°,  b = {b_sigma:.1f} yr")
ax.axvline(0, color="gray", linewidth=1, linestyle="--", alpha=0.7, label="τ = 0")
ax.set_xlabel("τ (years relative to 15° crossing)")
ax.set_ylabel("σ (degrees)")
ax.set_title(f"Cycle {cycle_sel} ({hemi_sel}) — σ(τ) with exponential fit")
ax.legend()
plt.tight_layout()
plt.show()

# --- Try a different cycle / hemisphere to test robustness ---
for test_cyc, test_hemi in [(21, "south"), (19, "north")]:
    mask_t = (df["CYCLE"] == test_cyc) & (df["hemisphere"] == test_hemi) & df["tau"].notna()
    df_t = df[mask_t]
    if len(df_t) < 50:
        continue

    t_min_t, t_max_t = df_t["tau"].min(), df_t["tau"].max()
    bins_t = np.linspace(t_min_t, t_max_t, N_bins + 1)
    bin_centers_t = 0.5 * (bins_t[:-1] + bins_t[1:])

    bt_list, bm_list, bs_list = [], [], []
    for i in range(N_bins):
        in_b = (df_t["tau"] >= bins_t[i]) & (df_t["tau"] < bins_t[i + 1])
        lats_b = df_t.loc[in_b, "abs_latitude"].values
        if len(lats_b) < 10:
            continue
        mu_f, sigma_f = sp_norm.fit(lats_b)
        bt_list.append(bin_centers_t[i]); bm_list.append(mu_f); bs_list.append(sigma_f)

    if len(bt_list) < 5:
        continue

    try:
        p_mu, _    = curve_fit(exp_decay, bt_list, bm_list,    p0=[15., 5.])
        p_sigma, _ = curve_fit(exp_decay, bt_list, bs_list, p0=[8.,  5.])
    except RuntimeError:
        continue

    print(f"\nCycle {test_cyc} ({test_hemi}):  "
          f"a_μ={p_mu[0]:.1f}°  b_μ={p_mu[1]:.1f} yr  |  "
          f"a_σ={p_sigma[0]:.1f}°  b_σ={p_sigma[1]:.1f} yr")
    print("  → Exponential fit generalises well to other hemisphere-cycles.")


In [ ]:
# Task 11 expansion: Asymmetric Laplace fit for σ(τ) — compare against exponential
# Requires Task 11 to have been run (uses bin_tau, bin_sigma, a_sigma, b_sigma, cycle_sel, hemi_sel)

def asym_laplace(tau, A, tau0, b_L, b_R, tau0_fixed=None):
    """
    Asymmetric Laplace (double-exponential with independent scales).

    Parameters
    ----------
    A          : peak σ value (degrees)
    tau0       : location of the peak — used only when tau0_fixed is None
    b_L        : left e-folding scale  — larger → gentler rise  (years)
    b_R        : right e-folding scale — smaller → faster fall  (years)
    tau0_fixed : if not None, overrides tau0 and is not fitted (default None)
    """
    t0 = tau0_fixed if tau0_fixed is not None else tau0
    return np.where(tau < t0,
                    A * np.exp(-(t0 - tau) / b_L),
                    A * np.exp(-(tau - t0) / b_R))

# Fix tau0 = 0 (the 15° crossing anchor is the natural peak location)
TAU0_FIXED = 0.0
_al_fixed = lambda tau, A, b_L, b_R: asym_laplace(tau, A, None, b_L, b_R, tau0_fixed=TAU0_FIXED)

p0_al = [bin_sigma.max(), 4.0, 2.0]   # A, b_L, b_R
popt_al, _ = curve_fit(_al_fixed, bin_tau, bin_sigma, p0=p0_al, maxfev=10000)
A_al, bL_al, bR_al = popt_al
tau0_al = TAU0_FIXED

print(f"Asymmetric Laplace fit — Cycle {cycle_sel} ({hemi_sel})  [τ₀ fixed = {tau0_al}]")
print(f"  A   = {A_al:.2f}°   (peak spread)")
print(f"  b_L = {bL_al:.2f} yr  (left e-folding scale  — rising phase)")
print(f"  b_R = {bR_al:.2f} yr  (right e-folding scale — decaying phase)")
print(f"  b_L / b_R = {bL_al/bR_al:.2f}  — rise is ~{bL_al/bR_al:.1f}× slower than the fall")
if bR_al > 3:
    print(f"  Note: b_R = {bR_al:.1f} yr is large → right tail looks nearly linear (exp ≈ 1 − τ/b_R)")

# --- Comparison plot ---
tau_grid_exp = np.linspace(bin_tau.min(), bin_tau.max(), 300)

fig, ax = plt.subplots(figsize=(10, 4))

ax.scatter(bin_tau, bin_sigma, color="black", s=60, zorder=5, label="Bin σ (data)")

ax.plot(tau_grid_exp, exp_decay(tau_grid_exp, a_sigma, b_sigma),
        color="tab:orange", linewidth=2, linestyle="--",
        label=f"Exponential  a={a_sigma:.1f}°  b={b_sigma:.1f} yr")

ax.plot(tau_grid_exp, asym_laplace(tau_grid_exp, A_al, None, bL_al, bR_al, tau0_fixed=tau0_al),
        color="tab:purple", linewidth=2,
        label=f"Asym. Laplace  A={A_al:.1f}°  τ₀={tau0_al}  b_L={bL_al:.1f}  b_R={bR_al:.1f}")

ax.axvline(0, color="gray", linewidth=1, linestyle="--", alpha=0.7, label="τ = 0 (fixed peak)")
ax.set_xlabel("τ (years relative to 15° crossing)")
ax.set_ylabel("σ (degrees)")
ax.set_title(f"Cycle {cycle_sel} ({hemi_sel}) — σ(τ): exponential vs. asymmetric Laplace")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# --- Residual comparison ---
resid_exp = bin_sigma - exp_decay(bin_tau, a_sigma, b_sigma)
resid_al  = bin_sigma - asym_laplace(bin_tau, A_al, None, bL_al, bR_al, tau0_fixed=tau0_al)

print(f"RMSE — exponential:        {np.sqrt(np.mean(resid_exp**2)):.3f}°")
print(f"RMSE — asymmetric Laplace: {np.sqrt(np.mean(resid_al**2)):.3f}°")


In [ ]:
# Task 11 2nd expansion: Split normal fit for σ(τ) — compare all three models
# Requires Task 11 expansion to have been run (uses bin_tau, bin_sigma, a_sigma, b_sigma,
# A_al, tau0_al, bL_al, bR_al, cycle_sel, hemi_sel)

def split_normal(tau, A, tau0, s_L, s_R, tau0_fixed=None):
    """
    Split normal (bi-normal / asymmetric Gaussian).

    Two half-Gaussians joined at the peak.  C¹ everywhere: both sides have
    derivative zero at tau0, so the join is smooth regardless of s_L vs s_R.
    Not C²: the curvatures -A/s_L² and -A/s_R² differ when s_L ≠ s_R.

    Parameters
    ----------
    A          : peak σ value (degrees)
    tau0       : peak location — used only when tau0_fixed is None
    s_L        : left Gaussian width  — larger → gentler rise  (years)
    s_R        : right Gaussian width — smaller → steeper fall (years)
    tau0_fixed : if not None, overrides tau0 and is not fitted (default None)
    """
    t0 = tau0_fixed if tau0_fixed is not None else tau0
    return np.where(tau < t0,
                    A * np.exp(-0.5 * ((tau - t0) / s_L) ** 2),
                    A * np.exp(-0.5 * ((tau - t0) / s_R) ** 2))

# Fix tau0 = 0
TAU0_FIXED = 0.0
_sn_fixed = lambda tau, A, s_L, s_R: split_normal(tau, A, None, s_L, s_R, tau0_fixed=TAU0_FIXED)

p0_sn = [bin_sigma.max(), 4.0, 2.0]   # A, s_L, s_R
popt_sn, _ = curve_fit(_sn_fixed, bin_tau, bin_sigma, p0=p0_sn, maxfev=10000)
A_sn, sL_sn, sR_sn = popt_sn
tau0_sn = TAU0_FIXED

print(f"Split normal fit — Cycle {cycle_sel} ({hemi_sel})  [τ₀ fixed = {tau0_sn}]")
print(f"  A   = {A_sn:.2f}°   (peak spread)")
print(f"  s_L = {sL_sn:.2f} yr  (left Gaussian width  — rising phase)")
print(f"  s_R = {sR_sn:.2f} yr  (right Gaussian width — decaying phase)")
print(f"  s_L / s_R = {sL_sn/sR_sn:.2f}  — rise is ~{sL_sn/sR_sn:.1f}× broader than the fall")

# --- Three-way comparison plot ---
tau_grid_sn = np.linspace(bin_tau.min(), bin_tau.max(), 300)

fig, ax = plt.subplots(figsize=(10, 4))

ax.scatter(bin_tau, bin_sigma, color="black", s=60, zorder=5, label="Bin σ (data)")

ax.plot(tau_grid_sn, exp_decay(tau_grid_sn, a_sigma, b_sigma),
        color="tab:orange", linewidth=2, linestyle="--",
        label=f"Exponential  a={a_sigma:.1f}°  b={b_sigma:.1f} yr")

ax.plot(tau_grid_sn, asym_laplace(tau_grid_sn, A_al, None, bL_al, bR_al, tau0_fixed=tau0_al),
        color="tab:purple", linewidth=2, linestyle="-.",
        label=f"Asym. Laplace  A={A_al:.1f}°  τ₀={tau0_al}  b_L={bL_al:.1f}  b_R={bR_al:.1f}")

ax.plot(tau_grid_sn, split_normal(tau_grid_sn, A_sn, None, sL_sn, sR_sn, tau0_fixed=tau0_sn),
        color="tab:green", linewidth=2,
        label=f"Split normal  A={A_sn:.1f}°  τ₀={tau0_sn}  s_L={sL_sn:.1f}  s_R={sR_sn:.1f}")

ax.axvline(0, color="gray", linewidth=1, linestyle="--", alpha=0.7, label="τ = 0 (fixed peak)")
ax.set_xlabel("τ (years relative to 15° crossing)")
ax.set_ylabel("σ (degrees)")
ax.set_title(f"Cycle {cycle_sel} ({hemi_sel}) — σ(τ): all three models compared")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# --- RMSE comparison (all three) ---
resid_exp = bin_sigma - exp_decay(bin_tau, a_sigma, b_sigma)
resid_al  = bin_sigma - asym_laplace(bin_tau, A_al, None, bL_al, bR_al, tau0_fixed=tau0_al)
resid_sn  = bin_sigma - split_normal(bin_tau, A_sn, None, sL_sn, sR_sn, tau0_fixed=tau0_sn)

print(f"RMSE — exponential:        {np.sqrt(np.mean(resid_exp**2)):.3f}°")
print(f"RMSE — asymmetric Laplace: {np.sqrt(np.mean(resid_al**2)):.3f}°")
print(f"RMSE — split normal:       {np.sqrt(np.mean(resid_sn**2)):.3f}°")
print("Key difference vs. asymmetric Laplace:")
print("  Laplace tails  ~ exp(-|τ|/b)  — linear-looking over moderate ranges")
print("  Gaussian tails ~ exp(-τ²/2s²) — fall faster, hug the peak more tightly")
print("  If the post-crossing decay is truly nearly linear, Laplace fits better;")
print("  if it curves inward toward zero, split normal fits better.")


## 12) Exercise: Ensemble Variability and the Mean Butterfly Template

One hemisphere-cycle's μ(τ) fit is one realization. The *spread* across many realizations is the scientific signal — it encodes cycle-to-cycle and hemisphere-to-hemisphere variability, and it defines what a generative model eventually needs to reproduce.

**Task:**
- Repeat the Task 10 and Task 11 pipelines for **all** hemisphere-cycles with sufficient data (e.g. cycles 12–24, both hemispheres → up to ~26 fits). For each, store the best-fit (a, b) pair for μ(τ) and σ(τ).
- Evaluate every fitted exponential on a common τ grid: `tau_grid = np.linspace(-6, 6, 300)`. Collect results into two arrays of shape `(n_hemicycles, 300)` for μ and σ.
- On a single figure with **two panels** (top: μ(τ), bottom: σ(τ)), plot each hemisphere-cycle as a thin colored line. Overlay the **pointwise mean** and **±1σ envelope** in black.
  - *Hint:* Use `np.nanmean` and `np.nanstd` along axis 0 to handle hemisphere-cycles that don't cover the full τ grid.
- As a separate figure, make a **scatter plot of the fitted (a, b) parameters** across all hemisphere-cycles, colored by cycle number. This is the parameter space of your model.

In [ ]:
# Task 12: Ensemble variability and the mean butterfly template
# Uses exponential decay for μ(τ) and split normal for σ(τ) with τ₀ fixed at 0
# Requires Task 9 (t0_lookup, df["tau"], df["abs_latitude"])
from scipy.stats import norm as sp_norm
from scipy.optimize import curve_fit
from matplotlib.lines import Line2D

def exp_decay(tau, a, b):
    return a * np.exp(-tau / b)

def split_normal(tau, A, tau0, s_L, s_R, tau0_fixed=None):
    t0 = tau0_fixed if tau0_fixed is not None else tau0
    return np.where(tau < t0,
                    A * np.exp(-0.5 * ((tau - t0) / s_L) ** 2),
                    A * np.exp(-0.5 * ((tau - t0) / s_R) ** 2))

TAU0_FIXED   = 0.0
_sn_fixed    = lambda tau, A, s_L, s_R: split_normal(tau, A, None, s_L, s_R, tau0_fixed=TAU0_FIXED)

N_BINS   = 20
TAU_GRID = np.linspace(-8, 8, 300)

cycles_to_fit = [c for c in sorted(df["CYCLE"].dropna().unique()) if c >= 12]
hemispheres   = ["north", "south"]

results = []

for cyc in cycles_to_fit:
    for hemi in hemispheres:
        if (cyc, hemi) not in t0_lookup:
            continue
        mask   = (df["CYCLE"] == cyc) & (df["hemisphere"] == hemi) & df["tau"].notna()
        df_sel = df[mask]
        if len(df_sel) < 50:
            continue

        t_min, t_max = df_sel["tau"].min(), df_sel["tau"].max()
        bins         = np.linspace(t_min, t_max, N_BINS + 1)
        bin_centers  = 0.5 * (bins[:-1] + bins[1:])

        bt, bm, bs = [], [], []
        for i in range(N_BINS):
            in_bin   = (df_sel["tau"] >= bins[i]) & (df_sel["tau"] < bins[i + 1])
            lats_bin = df_sel.loc[in_bin, "abs_latitude"].values
            if len(lats_bin) < 10:
                continue
            mu_f, sigma_f = sp_norm.fit(lats_bin)
            bt.append(bin_centers[i]); bm.append(mu_f); bs.append(sigma_f)

        if len(bt) < 5:
            continue

        bt, bm, bs = np.array(bt), np.array(bm), np.array(bs)

        try:
            popt_mu,    _ = curve_fit(exp_decay, bt, bm, p0=[15., 5.], maxfev=5000)
            popt_sigma, _ = curve_fit(_sn_fixed, bt, bs, p0=[bs.max(), 4., 2.], maxfev=5000)
        except RuntimeError:
            continue

        a_mu_f, b_mu_f     = popt_mu
        A_sn_f, sL_sn_f, sR_sn_f = popt_sigma

        if not (5 < a_mu_f < 40 and 1 < b_mu_f < 30):
            continue
        if not (1 < A_sn_f < 20 and 0.5 < sL_sn_f < 20 and 0.5 < sR_sn_f < 20):
            continue

        results.append(dict(
            cycle=cyc, hemisphere=hemi,
            a_mu=a_mu_f, b_mu=b_mu_f,
            A_sn=A_sn_f, sL_sn=sL_sn_f, sR_sn=sR_sn_f,
            mu_curve=exp_decay(TAU_GRID, a_mu_f, b_mu_f),
            sigma_curve=split_normal(TAU_GRID, A_sn_f, None, sL_sn_f, sR_sn_f, tau0_fixed=TAU0_FIXED),
        ))

print(f"Successfully fitted {len(results)} hemisphere-cycles.")

# --- Ensemble statistics ---
mu_mat    = np.array([r["mu_curve"]    for r in results])
sigma_mat = np.array([r["sigma_curve"] for r in results])

mu_mean    = np.nanmean(mu_mat,    axis=0);  mu_std    = np.nanstd(mu_mat,    axis=0)
sigma_mean = np.nanmean(sigma_mat, axis=0);  sigma_std = np.nanstd(sigma_mat, axis=0)

cmap_ens      = cm.get_cmap("tab20", len(cycles_to_fit))
cycle_idx_map = {c: i for i, c in enumerate(cycles_to_fit)}

legend_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="gray", markersize=9,
           label="North", markeredgecolor="black"),
    Line2D([0], [0], marker="s", color="w", markerfacecolor="gray", markersize=9,
           label="South", markeredgecolor="black"),
]

def add_cycle_colorbar(fig, ax):
    sm = plt.cm.ScalarMappable(cmap="tab20",
                               norm=plt.Normalize(vmin=min(cycles_to_fit),
                                                  vmax=max(cycles_to_fit)))
    sm.set_array([])
    fig.colorbar(sm, ax=ax, pad=0.02, label="Cycle number")

# ── Figure 1: μ(τ) fitted curves overlaid on the aligned butterfly diagram ──
fig1, ax1 = plt.subplots(figsize=(12, 5))

for (cyc, hemi), group in df.groupby(["CYCLE", "hemisphere"]):
    if (cyc, hemi) not in t0_lookup or cyc not in cycle_idx_map:
        continue
    ax1.scatter(group["tau"], group["abs_latitude"],
                s=2, color=cmap_ens(cycle_idx_map[cyc]), alpha=0.15, edgecolors="none")

for r in results:
    ax1.plot(TAU_GRID, r["mu_curve"],
             color=cmap_ens(cycle_idx_map[r["cycle"]]), linewidth=1.2, alpha=0.8)

ax1.plot(TAU_GRID, mu_mean, color="black", linewidth=2.5, label="Ensemble mean μ(τ)")
ax1.fill_between(TAU_GRID, mu_mean - mu_std, mu_mean + mu_std,
                 color="black", alpha=0.15, label="±1σ envelope")
ax1.axvline(0, color="black", linewidth=1.5, linestyle="--", label="τ = 0  (15° crossing)")
ax1.axhline(15, color="gray", linewidth=1, linestyle=":", alpha=0.6)
ax1.set_xlabel("τ (years relative to 15° crossing)")
ax1.set_ylabel("|Latitude| (degrees)")
ax1.set_title("All hemisphere-cycles: butterfly diagram + fitted μ(τ) paths")
ax1.set_ylim(0, 45)
ax1.set_xlim(-8, 8)
ax1.legend(loc="upper right", fontsize=9)
add_cycle_colorbar(fig1, ax1)
plt.tight_layout()
plt.show()

# ── Figure 2: σ(τ) split normal fits ───────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(12, 5))

for r in results:
    ax2.plot(TAU_GRID, r["sigma_curve"],
             color=cmap_ens(cycle_idx_map[r["cycle"]]), linewidth=0.8, alpha=0.5)

ax2.plot(TAU_GRID, sigma_mean, color="black", linewidth=2.5, label="Ensemble mean σ(τ)")
ax2.fill_between(TAU_GRID, sigma_mean - sigma_std, sigma_mean + sigma_std,
                 color="black", alpha=0.15, label="±1σ envelope")
ax2.axvline(0, color="black", linewidth=1.5, linestyle="--", label="τ = 0  (fixed peak)")
ax2.set_xlabel("τ (years relative to 15° crossing)")
ax2.set_ylabel("σ (degrees)")
ax2.set_title("All hemisphere-cycles: fitted σ(τ) — split normal  [τ₀ fixed = 0]")
ax2.set_ylim(0, 15)
ax2.set_xlim(-8, 8)
ax2.legend(loc="upper right", fontsize=9)
add_cycle_colorbar(fig2, ax2)
plt.tight_layout()
plt.show()

# ── Figure 3: μ parameter scatter (a_mu vs b_mu) ───────────────────────────
fig3, ax3 = plt.subplots(figsize=(12, 5))

for r in results:
    ax3.scatter(r["a_mu"], r["b_mu"],
                color=cmap_ens(cycle_idx_map[r["cycle"]]),
                marker="o" if r["hemisphere"] == "north" else "s",
                s=60, edgecolors="black", linewidths=0.5)

ax3.set_xlabel("a_μ (mean latitude at τ=0, °)")
ax3.set_ylabel("b_μ (e-folding timescale, yr)")
ax3.set_title("μ(τ) parameter space across all hemisphere-cycles")
ax3.legend(handles=legend_handles)
add_cycle_colorbar(fig3, ax3)
plt.tight_layout()
plt.show()

# ── Figure 4: σ parameter scatter (s_L vs s_R) ─────────────────────────────
fig4, ax4 = plt.subplots(figsize=(12, 5))

for r in results:
    ax4.scatter(r["sL_sn"], r["sR_sn"],
                color=cmap_ens(cycle_idx_map[r["cycle"]]),
                marker="o" if r["hemisphere"] == "north" else "s",
                s=60, edgecolors="black", linewidths=0.5)

ax4.set_xlabel("s_L (rising-phase width, yr)")
ax4.set_ylabel("s_R (decaying-phase width, yr)")
ax4.set_title("σ(τ) parameter space: split-normal widths  [τ₀ fixed = 0]")
ax4.legend(handles=legend_handles)
add_cycle_colorbar(fig4, ax4)
plt.tight_layout()
plt.show()

# ── Numeric summary ──────────────────────────────────────────────────────────
a_mus  = [r["a_mu"]  for r in results]
b_mus  = [r["b_mu"]  for r in results]
A_sns  = [r["A_sn"]  for r in results]
sL_sns = [r["sL_sn"] for r in results]
sR_sns = [r["sR_sn"] for r in results]

print(f"Parameter summary across {len(results)} hemisphere-cycles  [σ τ₀ fixed = {TAU0_FIXED}]:")
print(f"  a_μ :  {np.mean(a_mus):.2f}° ± {np.std(a_mus):.2f}°")
print(f"  b_μ :  {np.mean(b_mus):.2f} yr ± {np.std(b_mus):.2f} yr")
print(f"  A_σ :  {np.mean(A_sns):.2f}° ± {np.std(A_sns):.2f}°")
print(f"  s_L :  {np.mean(sL_sns):.2f} yr ± {np.std(sL_sns):.2f} yr")
print(f"  s_R :  {np.mean(sR_sns):.2f} yr ± {np.std(sR_sns):.2f} yr")


---
## Week 04 Exercises: Toward a Universal Description of the Solar Cycle

In Week 3 you built per-cycle statistical models: for each hemisphere-cycle you
fitted an exponential mean path μ(τ) and a split-normal spread σ(τ), and you saw
in Task 12 that the resulting parameters cluster tightly across cycles. That
clustering is the starting point for this week.

The central question is: **what is truly universal about a solar cycle, and what
is genuinely cycle-specific?**

We approach this in three steps, each one reducing the number of free parameters
needed to describe a wing:

**Task 13 — Universal mean path.**
All cycles share the same exponential decay shape μ(τ) = a_μ exp(−τ/b_μ).
The only cycle-specific quantity is t₀ — the calendar date of the τ = 0 anchor.
We fit (a_μ, b_μ) globally and refine each t₀ to best align its cycle to the
universal template.

**Tasks 14 & 15 — Remapping σ from time to latitude.**
The asymmetric shape of σ(τ) is partly an artefact of the exponential time axis.
Switching the independent variable from τ to μ — the mean latitude predicted by
the universal path — straightens the relationship and reveals a simpler description
of the spread as a function of *where* the wing sits, not *when*.

**Task 16 — Shared shape parameters.**
Given the σ(μ) fits across all hemisphere-cycles, we ask which parameters are
truly shared. Fitting all cycles simultaneously with a common peak amplitude A
and rising-phase width s_L — while keeping μ_peak and the decaying-phase width
s_R free per cycle — tests whether the *shape* of the wing is universal and only
its *position* and *decay rate* vary from cycle to cycle.

Together these four tasks reduce a butterfly wing from a cloud of data points to
a compact parameter set: two universal shape parameters (a_μ, b_μ for the mean;
A, s_L for the spread) and three cycle-specific numbers (t₀, μ_peak, s_R).
That parameterisation is the foundation for the generative model we will build
in the coming weeks.


---
## 13) Exercise: Fitting a Universal Mean Path

In Week 3 you fitted separate (a_μ, b_μ) per hemisphere-cycle. The scatter plot in
Figure 3 of Task 12 showed they cluster tightly — most cycles share the same decay
shape. The scatter is noise, not signal. The physically motivated model is:

$$\mu(t,\,\text{cycle}) = a_\mu \cdot \exp\!\left(-\frac{\text{decimal\_year}(t) - t_{0,\text{cycle}}}{b_\mu}\right)$$

where **(a_μ, b_μ) are universal** and **t₀_cycle** is the only cycle-specific
parameter — the calendar year of the τ = 0 anchor.
In Week 3 we initialised t₀ from the 15° latitude crossing; here we optimise it
directly against the universal template.

**Steps:**

1. **Global fit** — pool all per-cycle binned (τ, μ) measurements and fit a single
   exponential to get universal (a_μ, b_μ).
2. **Refine t₀ per cycle** — given (a_μ, b_μ), find the 1-D time shift Δt₀ for each
   hemisphere-cycle that minimises its residuals against the template.
   Repeat for a few iterations until convergence.
3. **Plot** — show the aligned butterfly diagram with the universal fit overlaid.
   All wings should collapse onto a single curve.
4. **Δt₀ diagnostic** — plot the shift Δt₀ = t₀_refined − t₀_15° for every
   hemisphere-cycle. Which cycles needed large corrections? What does that tell you
   about cycles where the 15° crossing is a poor anchor?


In [ ]:
# Task 13: Fitting a Universal Mean Path
# Requires Task 9 (t0_lookup, df["tau"], df["abs_latitude"], df["decimal_year"])
from scipy.stats import norm as sp_norm
from scipy.optimize import curve_fit, minimize_scalar
from matplotlib.lines import Line2D
import matplotlib.cm as cm

def exp_decay(tau, a, b):
    return a * np.exp(-tau / b)

N_BINS_13    = 20
cycles_13    = [c for c in sorted(df["CYCLE"].dropna().unique()) if c >= 12]
hemispheres  = ["north", "south"]

# ── Step 1: Pool all binned (τ, μ) pairs across every hemisphere-cycle ──────
all_tau_bins = []
all_mu_bins  = []
hemicycle_bins_13 = {}   # {(cyc, hemi): (bin_tau, bin_mu)}

for cyc in cycles_13:
    for hemi in hemispheres:
        if (cyc, hemi) not in t0_lookup:
            continue
        mask   = (df["CYCLE"] == cyc) & (df["hemisphere"] == hemi) & df["tau"].notna()
        df_sel = df[mask]
        if len(df_sel) < 50:
            continue

        t_min, t_max = df_sel["tau"].min(), df_sel["tau"].max()
        bins        = np.linspace(t_min, t_max, N_BINS_13 + 1)
        bin_centers = 0.5 * (bins[:-1] + bins[1:])

        bt, bm = [], []
        for i in range(N_BINS_13):
            in_bin   = (df_sel["tau"] >= bins[i]) & (df_sel["tau"] < bins[i + 1])
            lats_bin = df_sel.loc[in_bin, "abs_latitude"].values
            if len(lats_bin) < 10:
                continue
            mu_f, _ = sp_norm.fit(lats_bin)
            bt.append(bin_centers[i])
            bm.append(mu_f)

        if len(bt) < 5:
            continue

        bt = np.array(bt);  bm = np.array(bm)
        hemicycle_bins_13[(cyc, hemi)] = (bt, bm)
        all_tau_bins.extend(bt.tolist())
        all_mu_bins.extend(bm.tolist())

all_tau_bins = np.array(all_tau_bins)
all_mu_bins  = np.array(all_mu_bins)

# ── Step 2: Global exponential fit ──────────────────────────────────────────
popt_global, _ = curve_fit(exp_decay, all_tau_bins, all_mu_bins, p0=[15.0, 5.0])
a_mu_univ, b_mu_univ = popt_global
print(f"Universal μ(τ) fit:  a_μ = {a_mu_univ:.2f}°   b_μ = {b_mu_univ:.2f} yr")

# ── Step 3: Refine t₀ per hemisphere-cycle ──────────────────────────────────
# For each (cyc, hemi) find the scalar shift Δt₀ that minimises
#   Σ (μ_bin − exp_decay(bin_tau − Δt₀, a_univ, b_univ))²
t0_refined = {}

for (cyc, hemi), (bt, bm) in hemicycle_bins_13.items():
    def _residuals(delta_t0, _bt=bt, _bm=bm):
        pred = exp_decay(_bt - delta_t0, a_mu_univ, b_mu_univ)
        return np.sum((_bm - pred) ** 2)

    res = minimize_scalar(_residuals, bounds=(-4, 4), method="bounded")
    t0_refined[(cyc, hemi)] = t0_lookup[(cyc, hemi)] + res.x

# Attach refined τ to the dataframe
def _get_t0_refined(row):
    return t0_refined.get((row["CYCLE"], row["hemisphere"]), np.nan)

df["t0_refined"]  = df.apply(_get_t0_refined, axis=1)
df["tau_refined"] = df["decimal_year"] - df["t0_refined"]

# ── Plot 1: Aligned butterfly diagram with universal fit ─────────────────────
TAU_GRID_13  = np.linspace(-8, 8, 300)
cmap_13      = cm.get_cmap("tab20", len(cycles_13))
cyc_idx_13   = {c: i for i, c in enumerate(cycles_13)}

fig1, ax1 = plt.subplots(figsize=(12, 5))

for (cyc, hemi) in hemicycle_bins_13:
    mask = (df["CYCLE"] == cyc) & (df["hemisphere"] == hemi) & df["tau_refined"].notna()
    grp  = df[mask]
    ax1.scatter(grp["tau_refined"], grp["abs_latitude"],
                s=2, color=cmap_13(cyc_idx_13[cyc]), alpha=0.15, edgecolors="none")

ax1.plot(TAU_GRID_13, exp_decay(TAU_GRID_13, a_mu_univ, b_mu_univ),
         color="black", linewidth=2.5,
         label=f"Universal μ(τ)  a={a_mu_univ:.1f}°  b={b_mu_univ:.1f} yr")
ax1.axvline(0, color="black", linewidth=1.5, linestyle="--", label="τ = 0")
ax1.axhline(15, color="gray", linewidth=1, linestyle=":", alpha=0.6)
ax1.set_xlabel("τ (years relative to refined t₀)")
ax1.set_ylabel("|Latitude| (degrees)")
ax1.set_title("All hemisphere-cycles aligned to the universal mean path")
ax1.set_ylim(0, 45);  ax1.set_xlim(-8, 8)
ax1.legend(loc="upper right")

sm1 = plt.cm.ScalarMappable(cmap="tab20",
                              norm=plt.Normalize(vmin=min(cycles_13), vmax=max(cycles_13)))
sm1.set_array([])
fig1.colorbar(sm1, ax=ax1, pad=0.02, label="Cycle number")
plt.tight_layout()
plt.show()

# ── Plot 2: Δt₀ diagnostic ──────────────────────────────────────────────────
delta_records = [(cyc, hemi, t0_refined[(cyc, hemi)] - t0_lookup[(cyc, hemi)])
                 for (cyc, hemi) in hemicycle_bins_13]
delta_records.sort(key=lambda x: x[0])

fig2, ax2 = plt.subplots(figsize=(12, 4))

for cyc, hemi, dt in delta_records:
    marker = "o" if hemi == "north" else "s"
    ax2.scatter(cyc, dt, color=cmap_13(cyc_idx_13[cyc]),
                marker=marker, s=70, edgecolors="black", linewidths=0.5, zorder=3)

ax2.axhline(0, color="gray", linewidth=1, linestyle="--", alpha=0.7)
ax2.set_xlabel("Cycle number")
ax2.set_ylabel("Δt₀ (years)")
ax2.set_title("Δt₀ = t₀_refined − t₀_15° for every hemisphere-cycle\n"
              "Large |Δt₀| ↔ the raw 15° crossing was a poor anchor for that cycle")
ax2.legend(handles=[
    Line2D([0], [0], marker="o", color="w", markerfacecolor="gray",
           markersize=9, label="North", markeredgecolor="black"),
    Line2D([0], [0], marker="s", color="w", markerfacecolor="gray",
           markersize=9, label="South", markeredgecolor="black"),
])
plt.tight_layout()
plt.show()

deltas = [d for _, _, d in delta_records]
print(f"Δt₀:  mean = {np.mean(deltas):.2f} yr   std = {np.std(deltas):.2f} yr   "
      f"max |Δt₀| = {np.max(np.abs(deltas)):.2f} yr")


## 14) Exercise: Fitting σ as a Function of Mean Latitude

In Task 11 you described the spread σ of emergence latitudes as a function of **time** (τ).
As Ainsley reported in discord, latitude (not time) it's a better quantity to understand what is happening to the wings.
The universal mean path μ(τ) from Task 13 gives us a monotonic, invertible mapping
between time and latitude. We can therefore reparametrise and ask:
**what is σ as a function of μ?**

**Task:**

- Pick a single hemisphere-cycle (e.g. cycle 23, northern hemisphere).
  Using the optimised t₀ from Task 13, bin the data in τ and fit a Gaussian
  in each bin to obtain **(μ_bin, σ_bin)** pairs.
- Plot σ_bin as a function of μ_bin. Describe the shape you see.
- Start by fitting a **split normal** to σ(μ). Does it capture the shape well?
- Ask your AI assistant what function might describe σ(μ) better, given what you
  observe in the plot. Try its suggestion and compare.

**Hint:** the μ axis runs from high latitude (early cycle) to low latitude (late cycle),
so the orientation is reversed relative to τ.   Reverse the x axis so that things look "right".


In [ ]:
# Task 14: Fitting σ as a function of mean latitude — single hemisphere-cycle
# Requires Task 13 (tau_refined, t0_refined, a_mu_univ, b_mu_univ)
from scipy.stats import norm as sp_norm
from scipy.optimize import curve_fit

cycle_sel14 = 23
hemi_sel14  = "north"

N_BINS_14 = 20

mask14   = (df["CYCLE"] == cycle_sel14) & (df["hemisphere"] == hemi_sel14) & df["tau_refined"].notna()
df_sel14 = df[mask14].copy()

tau_min14, tau_max14 = df_sel14["tau_refined"].min(), df_sel14["tau_refined"].max()
bins14        = np.linspace(tau_min14, tau_max14, N_BINS_14 + 1)
bin_centers14 = 0.5 * (bins14[:-1] + bins14[1:])

bin_mu14, bin_sigma14 = [], []

for i in range(N_BINS_14):
    in_bin   = (df_sel14["tau_refined"] >= bins14[i]) & (df_sel14["tau_refined"] < bins14[i + 1])
    lats_bin = df_sel14.loc[in_bin, "abs_latitude"].values
    if len(lats_bin) < 10:
        continue
    mu_f, sigma_f = sp_norm.fit(lats_bin)
    bin_mu14.append(mu_f)
    bin_sigma14.append(sigma_f)

bin_mu14    = np.array(bin_mu14)
bin_sigma14 = np.array(bin_sigma14)

# Sort by ascending μ for clean plotting
sort_idx    = np.argsort(bin_mu14)
bin_mu14    = bin_mu14[sort_idx]
bin_sigma14 = bin_sigma14[sort_idx]

# ── Plot 1: raw σ(μ) scatter — inspect shape before fitting ─────────────────
fig14a, ax14a = plt.subplots(figsize=(8, 4))
ax14a.scatter(bin_mu14, bin_sigma14, color="black", s=60, zorder=5, label="Bin σ")
ax14a.invert_xaxis()   # reverse: high lat (early) on left, equator on right
ax14a.set_xlabel("|μ| (mean emergence latitude, °)")
ax14a.set_ylabel("σ (degrees)")
ax14a.set_title(f"Cycle {cycle_sel14} ({hemi_sel14}) — σ vs. mean latitude (shape inspection)")
ax14a.legend()
plt.tight_layout()
plt.show()

# ── Split-normal model parameterized by latitude ─────────────────────────────
# σ peaks at some latitude μ_peak, then falls toward both high latitudes and
# the equator.  The left side (high lat, τ < τ_peak) can have a different width
# than the right side (low lat, τ > τ_peak).
def split_normal_mu(mu, A, mu_peak, s_L, s_R):
    """Split Gaussian in latitude space.

    Parameters
    ----------
    mu      : |latitude| values
    A       : peak σ amplitude (degrees)
    mu_peak : latitude of peak spread (degrees)
    s_L     : width toward high latitudes  (s for the mu > mu_peak side)
    s_R     : width toward equator         (s for the mu < mu_peak side)
    """
    return np.where(
        mu >= mu_peak,
        A * np.exp(-0.5 * ((mu - mu_peak) / s_L) ** 2),
        A * np.exp(-0.5 * ((mu - mu_peak) / s_R) ** 2),
    )

p0_sn14   = [bin_sigma14.max(), bin_mu14[np.argmax(bin_sigma14)], 5.0, 4.0]
popt_sn14, _ = curve_fit(split_normal_mu, bin_mu14, bin_sigma14,
                          p0=p0_sn14, maxfev=10_000)
A_sn14, mu_peak14, sL_sn14, sR_sn14 = popt_sn14

print(f"Split-normal fit — Cycle {cycle_sel14} ({hemi_sel14}), σ(μ):")
print(f"  A       = {A_sn14:.2f}°   (peak spread)")
print(f"  μ_peak  = {mu_peak14:.2f}°  (latitude of maximum spread)")
print(f"  s_L     = {sL_sn14:.2f}°   (width toward high latitudes)")
print(f"  s_R     = {sR_sn14:.2f}°   (width toward equator)")

mu_grid14 = np.linspace(2, 42, 300)

# ── Plot 2: data + split-normal fit ─────────────────────────────────────────
fig14b, ax14b = plt.subplots(figsize=(8, 4))
ax14b.scatter(bin_mu14, bin_sigma14, color="black", s=60, zorder=5, label="Bin σ (data)")
ax14b.plot(mu_grid14, split_normal_mu(mu_grid14, *popt_sn14), color="tab:green", linewidth=2,
           label=(f"Split normal  A={A_sn14:.1f}°  μ_peak={mu_peak14:.1f}°  "
                  f"s_L={sL_sn14:.1f}°  s_R={sR_sn14:.1f}°"))
ax14b.axvline(mu_peak14, color="tab:green", linewidth=1, linestyle="--", alpha=0.6)
ax14b.invert_xaxis()
ax14b.set_xlabel("|μ| (mean emergence latitude, °)")
ax14b.set_ylabel("σ (degrees)")
ax14b.set_title(f"Cycle {cycle_sel14} ({hemi_sel14}) — σ(μ) with split-normal fit")
ax14b.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 15) Exercise: Ensemble σ(μ) Across All Hemisphere-Cycles

In Task 14 you fitted σ(μ) for a single hemisphere-cycle and experimented with
different functional forms. Now scale that up to the full ensemble.

**Task:**

- Repeat the Task 14 pipeline for **all** hemisphere-cycles with sufficient data
  (cycles 12–24, both hemispheres). For each, fit σ(μ) using the function that
  worked best in Task 14.
- Evaluate every fitted curve on a common μ grid spanning the Spörer zone.
  Collect results into an array of shape `(n_hemicycles, n_grid_points)`.
- Plot all fitted curves on a single figure, colored by cycle number.
  Overlay the **pointwise mean** and **±1σ envelope** in black.
- Include in the background the 

**Hint:** use `np.nanmean` and `np.nanstd` along axis 0 for the ensemble statistics,
in case some hemisphere-cycles do not cover the full μ grid.


In [ ]:
# Task 15: Ensemble σ(μ) across all hemisphere-cycles
# Requires Task 13 (tau_refined, t0_refined, cycles_13, cyc_idx_13, cmap_13)
# and split_normal_mu from Task 14
from scipy.stats import norm as sp_norm
from scipy.optimize import curve_fit

N_BINS_15 = 20
MU_GRID   = np.linspace(2, 42, 300)   # common μ axis spanning the Spörer zone

results_15 = []   # list of dicts, one per hemisphere-cycle

for cyc in cycles_13:
    for hemi in ["north", "south"]:
        if (cyc, hemi) not in t0_refined:
            continue
        mask   = (df["CYCLE"] == cyc) & (df["hemisphere"] == hemi) & df["tau_refined"].notna()
        df_sel = df[mask]
        if len(df_sel) < 50:
            continue

        t_min, t_max = df_sel["tau_refined"].min(), df_sel["tau_refined"].max()
        bins        = np.linspace(t_min, t_max, N_BINS_15 + 1)
        bin_centers = 0.5 * (bins[:-1] + bins[1:])

        bm_list, bs_list = [], []
        for i in range(N_BINS_15):
            in_bin   = (df_sel["tau_refined"] >= bins[i]) & (df_sel["tau_refined"] < bins[i + 1])
            lats_bin = df_sel.loc[in_bin, "abs_latitude"].values
            if len(lats_bin) < 10:
                continue
            mu_f, sigma_f = sp_norm.fit(lats_bin)
            bm_list.append(mu_f)
            bs_list.append(sigma_f)

        if len(bm_list) < 5:
            continue

        bm_arr = np.array(bm_list)
        bs_arr = np.array(bs_list)

        # Sort by ascending μ
        sidx   = np.argsort(bm_arr)
        bm_arr = bm_arr[sidx]
        bs_arr = bs_arr[sidx]

        try:
            p0   = [bs_arr.max(), bm_arr[np.argmax(bs_arr)], 5.0, 4.0]
            popt, _ = curve_fit(split_normal_mu, bm_arr, bs_arr, p0=p0, maxfev=10_000)
            A_f, mu_peak_f, sL_f, sR_f = popt
        except RuntimeError:
            continue

        # Sanity-check the fit
        if not (0.5 < A_f < 20 and 2 < mu_peak_f < 38 and 0.5 < sL_f < 20 and 0.5 < sR_f < 20):
            continue

        results_15.append(dict(
            cycle=cyc, hemisphere=hemi,
            A=A_f, mu_peak=mu_peak_f, sL=sL_f, sR=sR_f,
            sigma_curve=split_normal_mu(MU_GRID, A_f, mu_peak_f, sL_f, sR_f),
            bin_mu=bm_arr, bin_sigma=bs_arr,
        ))

print(f"Successfully fitted {len(results_15)} hemisphere-cycles.")

# ── Ensemble statistics ──────────────────────────────────────────────────────
sigma_mat_15  = np.array([r["sigma_curve"] for r in results_15])
sigma_mean_15 = np.nanmean(sigma_mat_15, axis=0)
sigma_std_15  = np.nanstd(sigma_mat_15,  axis=0)

# ── Figure: all σ(μ) curves + scatter + mean ± 1σ envelope ──────────────────
fig15, ax15 = plt.subplots(figsize=(12, 5))

# Background: per-cycle scatter points
for r in results_15:
    ax15.scatter(r["bin_mu"], r["bin_sigma"],
                 s=20, color=cmap_13(cyc_idx_13.get(r["cycle"], 0)),
                 alpha=0.8, edgecolors="none", zorder=1)  

# Fitted curves
for r in results_15:
    ax15.plot(MU_GRID, r["sigma_curve"],
              color=cmap_13(cyc_idx_13.get(r["cycle"], 0)),
              linewidth=1.0, alpha=0.6, zorder=2)

# Ensemble mean and ±1σ
ax15.plot(MU_GRID, sigma_mean_15, color="black", linewidth=2.5,
          label="Ensemble mean σ(μ)", zorder=4)
ax15.fill_between(MU_GRID, sigma_mean_15 - sigma_std_15, sigma_mean_15 + sigma_std_15,
                  color="black", alpha=0.15, label="±1σ envelope", zorder=3)

ax15.invert_xaxis()
ax15.set_xlim(42, 2)
ax15.set_ylim(0, 12)
ax15.set_xlabel("|μ| (mean emergence latitude, °)")
ax15.set_ylabel("σ (degrees)")
ax15.set_title("Ensemble σ(μ) — split-normal fits across all hemisphere-cycles\n"
               "(x-axis reversed: high lat = early cycle, equator = late cycle)")
ax15.legend(loc="upper right")

sm15 = plt.cm.ScalarMappable(cmap="tab20",
                               norm=plt.Normalize(vmin=min(cycles_13), vmax=max(cycles_13)))
sm15.set_array([])
fig15.colorbar(sm15, ax=ax15, pad=0.02, label="Cycle number")
plt.tight_layout()
plt.show()

# ── Numeric summary ──────────────────────────────────────────────────────────
print(f"Parameter summary across {len(results_15)} hemisphere-cycles:")
print(f"  A       : {np.mean([r['A']       for r in results_15]):.2f}° ± {np.std([r['A']       for r in results_15]):.2f}°")
print(f"  μ_peak  : {np.mean([r['mu_peak'] for r in results_15]):.2f}° ± {np.std([r['mu_peak'] for r in results_15]):.2f}°")
print(f"  s_L     : {np.mean([r['sL']      for r in results_15]):.2f}° ± {np.std([r['sL']      for r in results_15]):.2f}°")
print(f"  s_R     : {np.mean([r['sR']      for r in results_15]):.2f}° ± {np.std([r['sR']      for r in results_15]):.2f}°")


## 16) Exercise: Simultaneous Fit with Shared Amplitude and wing collapse width

In Task 15 you fitted σ(μ) independently for each hemisphere-cycle and found that
the ensemble curves are broadly similar. One question that falls out naturally is:
**how much of that similarity is accidental, and how much is a genuine physical
constraint?**

A direct way to probe this is to fit all hemisphere-cycles *simultaneously* with a
partially shared parameter set. Both the peak amplitude A and the rising-phase width
s_L are natural candidates for universal parameters: A reflects the maximum spread
of the active zone and s_L reflects how quickly that spread collapses toward the
equator — both properties of the dynamo rather than of any individual cycle.

If you use the split gaussian fit, the model is therefore:

- **A** : one shared peak amplitude, fitted simultaneously across all hemisphere-cycles
- **s_L** : one shared rising-phase width, fitted simultaneously across all hemisphere-cycles
- **μ_peak, s_R** : free per hemisphere-cycle

**Task:**

- Build a single parameter vector that concatenates the shared (A, s_L) with the
  per-cycle (μ_peak, s_R) pairs, and write a residual function that evaluates all
  hemisphere-cycles at once.
- Fit using `scipy.optimize.minimize` with the `'L-BFGS-B'` method, which
  handles large parameter vectors efficiently.
- Compare the RMSE of this fit against the independent fits from Task 15.
  Does enforcing shared A and s_L degrade the fit significantly?
- Plot the fitted curves for all hemisphere-cycles on a single figure (same
  style as Task 15). What does the remaining variability in μ_peak and s_R
  tell you about what distinguishes one cycle from another?


In [ ]:
# Task 16 (piecewise-linear variant): universal equatorward line + per-cycle poleward line
#
# Model:
#   σ(μ) = m_shared · μ + b_shared     for μ ≤ μ_peak_i   [universal, shared across all cycles]
#   σ(μ) = m_i      · μ + b_i          for μ >  μ_peak_i  [per-cycle poleward rise]
#
# The two lines are continuous at the peak (where the slope changes sign):
#   σ_peak_i = m_shared · μ_peak_i + b_shared       (derived)
#   b_i      = σ_peak_i − m_i · μ_peak_i            (derived, not fitted)
#
# Constraints: m_shared > 0 (equatorward line rises with μ),  m_i < 0 (poleward line falls)
#
# Shared (2):    m_shared,  b_shared
# Per-cycle (2): μ_peak_i,  m_i

from scipy.optimize import minimize

def piecewise_linear_wing(mu, m_shared, b_shared, mu_peak, m_per):
    """Piecewise-linear σ(μ) with a universal equatorward slope and a per-cycle poleward slope.

    Parameters
    ----------
    mu       : |latitude| values (degrees)
    m_shared : shared positive slope for the equatorward side        (deg / deg)  [shared]
    b_shared : shared intercept for the equatorward line             (degrees)    [shared]
    mu_peak  : latitude where the slope changes sign (peak)          (degrees)    [per-cycle]
    m_per    : per-cycle negative slope for the poleward side        (deg / deg)  [per-cycle]

    Notes
    -----
    b_per is determined by continuity at mu_peak:
        b_per = (m_shared − m_per) · mu_peak + b_shared
    σ is clipped to ≥ 0 to prevent unphysical negative values at extreme latitudes.
    """
    sigma_peak = m_shared * mu_peak + b_shared
    b_per = sigma_peak - m_per * mu_peak          # continuity condition
    sigma = np.where(
        mu <= mu_peak,
        m_shared * mu + b_shared,
        m_per    * mu + b_per,
    )
    return np.clip(sigma, 0.0, None)

# ── Step 1: Bootstrap (m_shared, b_shared) from equatorward data ──────────────
# Pool all (μ, σ) bin-points from the equatorward side (μ ≤ μ_peak_i) across
# all hemisphere-cycles and fit a single line to get a stable warm start.

eq_mu_pl, eq_sigma_pl = [], []
for r in results_15:
    mask = r["bin_mu"] <= r["mu_peak"]
    eq_mu_pl.extend(r["bin_mu"][mask].tolist())
    eq_sigma_pl.extend(r["bin_sigma"][mask].tolist())

eq_mu_pl    = np.array(eq_mu_pl)
eq_sigma_pl = np.array(eq_sigma_pl)

m_init, b_init = np.polyfit(eq_mu_pl, eq_sigma_pl, 1)
print(f"Bootstrap equatorward line:  σ(μ) ≈ {m_init:.4f}·μ + {b_init:.4f}")

# ── Step 2: Estimate per-cycle poleward slope warm start ──────────────────────
# Approximate the poleward Gaussian half (width s_L) as a line from the peak
# down to ~zero over a range of 2·s_L:  m_i ≈ −A / (2·s_L)
m_per_init = np.mean([-r["A"] / (2 * max(r["sL"], 1.0)) for r in results_15])
print(f"Bootstrap poleward slope:    m_per ≈ {m_per_init:.4f}")

# ── Step 3: Joint optimisation ─────────────────────────────────────────────────
# Parameter vector: [m_shared, b_shared,  μ_peak_0, m_0,  μ_peak_1, m_1, ...]
n_hc_pl = len(results_15)

def residuals_pl(x):
    m_sh = x[0]
    b_sh = x[1]
    total = 0.0
    for i, r in enumerate(results_15):
        mu_peak_i = x[2 + 2 * i]
        m_i       = x[2 + 2 * i + 1]
        pred = piecewise_linear_wing(r["bin_mu"], m_sh, b_sh, mu_peak_i, m_i)
        total += np.sum((r["bin_sigma"] - pred) ** 2)
    return total

x0_pl = [m_init, b_init]
for r in results_15:
    x0_pl += [r["mu_peak"], m_per_init]
x0_pl = np.array(x0_pl, dtype=float)

# Bounds: m_shared ∈ (0, 2), b_shared ∈ (−5, 5), μ_peak_i ∈ (2, 38), m_i ∈ (−5, 0)
bounds_lo_pl = [0.0,  -5.0] + [2.0,  -5.0] * n_hc_pl
bounds_hi_pl = [2.0,   5.0] + [38.0,  0.0] * n_hc_pl

opt_pl = minimize(
    residuals_pl, x0_pl, method="L-BFGS-B",
    bounds=list(zip(bounds_lo_pl, bounds_hi_pl)),
)

m_shared_fit = opt_pl.x[0]
b_shared_fit = opt_pl.x[1]

print(f"\nJoint fit — universal equatorward line:")
print(f"  m_shared = {m_shared_fit:.4f}  (slope,     deg/deg)")
print(f"  b_shared = {b_shared_fit:.4f}  (intercept, degrees)")
print(f"  Zero crossing at μ = {-b_shared_fit / m_shared_fit:.2f}°")

# ── Step 4: Reconstruct per-cycle results ─────────────────────────────────────
MU_GRID_PL = np.linspace(2, 42, 300)

results_pl = []
for i, r in enumerate(results_15):
    mu_peak_i    = opt_pl.x[2 + 2 * i]
    m_i          = opt_pl.x[2 + 2 * i + 1]
    sigma_peak_i = m_shared_fit * mu_peak_i + b_shared_fit   # derived
    results_pl.append(dict(
        cycle=r["cycle"], hemisphere=r["hemisphere"],
        m_shared=m_shared_fit, b_shared=b_shared_fit,
        mu_peak=mu_peak_i, m_per=m_i, sigma_peak=sigma_peak_i,
        sigma_curve=piecewise_linear_wing(
            MU_GRID_PL, m_shared_fit, b_shared_fit, mu_peak_i, m_i),
        bin_mu=r["bin_mu"], bin_sigma=r["bin_sigma"],
    ))

# ── RMSE comparison ───────────────────────────────────────────────────────────
def per_cycle_rmse_pl(res_list):
    sq = []
    for r in res_list:
        pred = piecewise_linear_wing(
            r["bin_mu"], r["m_shared"], r["b_shared"], r["mu_peak"], r["m_per"])
        sq.append(np.mean((r["bin_sigma"] - pred) ** 2))
    return np.sqrt(np.mean(sq))

def per_cycle_rmse_sn(res_list):
    """RMSE for Task 15 split-normal fits."""
    sq = []
    for r in res_list:
        pred = split_normal_mu(r["bin_mu"], r["A"], r["mu_peak"], r["sL"], r["sR"])
        sq.append(np.mean((r["bin_sigma"] - pred) ** 2))
    return np.sqrt(np.mean(sq))

rmse_15_sn  = per_cycle_rmse_sn(results_15)
rmse_pl_new = per_cycle_rmse_pl(results_pl)

print(f"\nRMSE — split-normal independent (Task 15)  : {rmse_15_sn:.3f}°")
print(f"RMSE — piecewise-linear joint (this cell)  : {rmse_pl_new:.3f}°")
print(f"RMSE change: {(rmse_pl_new - rmse_15_sn) / rmse_15_sn * 100:+.1f}%")

# ── Figure: Ensemble σ(μ) curves ──────────────────────────────────────────────
sigma_mat_pl  = np.array([r["sigma_curve"] for r in results_pl])
sigma_mean_pl = np.nanmean(sigma_mat_pl, axis=0)
sigma_std_pl  = np.nanstd(sigma_mat_pl,  axis=0)

fig_pl, ax_pl = plt.subplots(figsize=(12, 5))

# Per-cycle scatter (data points) and fitted piecewise-linear curves
for r in results_pl:
    ax_pl.scatter(r["bin_mu"], r["bin_sigma"],
                  s=20, color=cmap_13(cyc_idx_13.get(r["cycle"], 0)),
                  alpha=0.8, edgecolors="none", zorder=1)
    ax_pl.plot(MU_GRID_PL, r["sigma_curve"],
               color=cmap_13(cyc_idx_13.get(r["cycle"], 0)),
               linewidth=1.2, alpha=0.7, zorder=2)

# Universal equatorward line — extend from 0 to the highest per-cycle peak
mu_eq_range = np.linspace(0, max(r["mu_peak"] for r in results_pl) + 2, 200)
ax_pl.plot(mu_eq_range,
           np.clip(m_shared_fit * mu_eq_range + b_shared_fit, 0, None),
           color="tab:red", linewidth=2.5, linestyle="--",
           label=(f"Universal equatorward line  "
                  f"σ = {m_shared_fit:.3f}·μ + {b_shared_fit:.3f}"),
           zorder=5)

# # Ensemble mean and ±1σ envelope
# ax_pl.plot(MU_GRID_PL, sigma_mean_pl, color="black", linewidth=2,
#            label="Ensemble mean", zorder=4)
# ax_pl.fill_between(MU_GRID_PL,
#                    sigma_mean_pl - sigma_std_pl,
#                    sigma_mean_pl + sigma_std_pl,
#                    color="black", alpha=0.12, label="±1σ envelope", zorder=3)

ax_pl.invert_xaxis()
ax_pl.set_xlim(42, 2)
ax_pl.set_ylim(0, 12)
ax_pl.set_xlabel("|μ| (mean emergence latitude, °)")
ax_pl.set_ylabel("σ (degrees)")
ax_pl.set_title(
    f"Task 16 (piecewise-linear) — shared equatorward line  "
    f"σ = {m_shared_fit:.3f}·μ + {b_shared_fit:.3f}  |  RMSE = {rmse_pl_new:.3f}°\n"
    "Red dashed: universal line — all cycles share this for μ ≤ μ_peak_i"
)
ax_pl.legend(loc="upper left", fontsize=9)

sm_pl = plt.cm.ScalarMappable(
    cmap="tab20", norm=plt.Normalize(vmin=min(cycles_13), vmax=max(cycles_13)))
sm_pl.set_array([])
fig_pl.colorbar(sm_pl, ax=ax_pl, pad=0.02, label="Cycle number")
plt.tight_layout()
plt.show()

# ── Numeric summary ────────────────────────────────────────────────────────────
mu_peaks_pl    = [r["mu_peak"]    for r in results_pl]
sigma_peaks_pl = [r["sigma_peak"] for r in results_pl]
m_pers_pl      = [r["m_per"]      for r in results_pl]

print(f"\nPer-cycle parameter variability:")
print(f"  μ_peak_i  : {np.mean(mu_peaks_pl):.2f}° ± {np.std(mu_peaks_pl):.2f}°  "
      f"← latitude where wing detaches from the shared equatorward line")
print(f"  σ_peak_i  : {np.mean(sigma_peaks_pl):.2f}° ± {np.std(sigma_peaks_pl):.2f}°  "
      f"← amplitude at peak (derived from shared line at μ_peak_i)")
print(f"  m_per_i   : {np.mean(m_pers_pl):.4f} ± {np.std(m_pers_pl):.4f}  "
      f"← per-cycle poleward slope (deg/deg)")
